In [2]:
"""
Genera el PARTE DIARIO POR PROPIEDAD (servicio de control biologico) en Excel,
una hoja por cada codigo de propiedad, con los lotes en estado PLANIFICADO
de la tabla drones_control_bio.parte_diario_ctrl_bio.

Uso (Jupyter): pega todo en una celda, edita la lista CODIGOS al final
y ejecuta. Para generar otro Excel basta con llamar:
    generar([122, 2238])
    generar([122, 2238], "partes_semana45.xlsx")

Requisitos (una sola vez, en una celda):
    %pip install psycopg2-binary openpyxl
"""

import datetime as dt
import os
import re

from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.worksheet.datavalidation import DataValidation

# ---------------------------------------------------------------------------
# CONEXION A LA BASE (edita estos valores o define las variables de entorno)
# ---------------------------------------------------------------------------
DB = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5433")),
    "dbname": os.getenv("PGDATABASE", "utea"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "77663540"),
}

ESTADO = "PLANIFICADO"
FILAS_EXTRA = 3  # filas vacias al final del detalle para lotes no previstos

SQL = """
SELECT unidad_01, unidad_02, unidad_03, unidad_04, unidad_05,
       area, producto_plan, producto_dosis, os, semana_plan
FROM drones_control_bio.parte_diario_ctrl_bio
WHERE estado = %s AND unidad_01 = ANY(%s)
ORDER BY unidad_01, unidad_05
"""

# ---------------------------------------------------------------------------
# ESTILOS (copiados de la plantilla original)
# ---------------------------------------------------------------------------
VERDE = PatternFill("solid", fgColor="1F4E3D")
VERDE_CLARO = PatternFill("solid", fgColor="E2EFDA")
BLANCO_B = Font(bold=True, color="FFFFFF", size=11)
NEGRO_B = Font(bold=True, size=11)
AZUL = Font(color="0000FF", size=11)
DELGADO = Side(style="thin")
BORDE = Border(left=DELGADO, right=DELGADO, top=DELGADO, bottom=DELGADO)
CENTRO = Alignment(horizontal="center", vertical="center", wrap_text=True)
DERECHA = Alignment(horizontal="right", vertical="center")
IZQ = Alignment(horizontal="left", vertical="center")
IZQ_ARRIBA = Alignment(horizontal="left", vertical="top", wrap_text=True)
NUM = "#,##0.00"

ANCHOS = {"A": 1.86, "B": 4.71, "C": 13.43, "D": 13.71, "E": 20.14,
          "F": 27.43, "G": 14.29, "H": 18.0}


def nombre_producto(p):
    p = (p or "").strip().lower()
    if p.startswith("tricho"):
        return "Trichograma"
    if p.startswith("criso"):
        return "Crisopa"
    return p.capitalize()


def rango(ws, ref, valor=None, font=None, fill=None, align=None,
          borde=True, fmt=None):
    """Combina un rango, pone valor y aplica estilo a todas sus celdas."""
    if ":" in ref:
        ws.merge_cells(ref)
    celdas = ws[ref]
    if not isinstance(celdas, tuple):
        celdas = ((celdas,),)
    elif not isinstance(celdas[0], tuple):
        celdas = (celdas,)
    for fila in celdas:
        for c in fila:
            if borde:
                c.border = BORDE
            if fill:
                c.fill = fill
            if font:
                c.font = font
            if align:
                c.alignment = align
            if fmt:
                c.number_format = fmt
    if valor is not None:
        celdas[0][0].value = valor


def titulo_hoja(codigo, nombre, usados):
    base = re.sub(r"[\[\]:*?/\\]", "", f"{codigo} - {nombre}")[:31].strip()
    t, i = base, 2
    while t.lower() in usados:
        suf = f" ({i})"
        t, i = base[:31 - len(suf)] + suf, i + 1
    usados.add(t.lower())
    return t


def crear_hoja(wb, lotes, usados):
    p = lotes[0]
    ws = wb.create_sheet(titulo_hoja(p["unidad_01"], p["unidad_02"], usados))
    for col, w in ANCHOS.items():
        ws.column_dimensions[col].width = w

    # --- Titulo -----------------------------------------------------------
    rango(ws, "B1:H1", "PARTE DIARIO SERVICIO DE CONTROL BIOLÓGICO",
          BLANCO_B, VERDE, CENTRO, borde=False)
    ws.row_dimensions[1].height = 27.95

    n = len(lotes)
    ini = 12                         # primera fila del detalle
    fin = ini + n + FILAS_EXTRA - 1  # ultima fila del detalle
    f_cri, f_tri, f_tot = fin + 1, fin + 2, fin + 3

    # --- Encabezado -------------------------------------------------------
    def etiqueta(ref, txt):
        rango(ws, ref, txt, NEGRO_B, VERDE_CLARO, IZQ)

    def dato(ref, val=None, fmt=None):
        rango(ws, ref, val, AZUL, None, CENTRO, fmt=fmt)

    etiqueta("B3:C3", "Piloto:");               dato("D3:E3")
    etiqueta("F3", "Zona:");                    dato("G3:H3")
    etiqueta("B4:C4", "Código Propiedad:");     dato("D4:E4", p["unidad_01"])
    etiqueta("F4", "OS / Semana plan:")
    dato("G4:H4", f"{p['os'] or '-'} / {p['semana_plan'] or '-'}")
    etiqueta("B5:C5", "Nombre Propiedad:");     dato("D5:H5", p["unidad_02"])
    etiqueta("B6:C6", "Propietario:");          dato("D6:H6", p["unidad_04"])
    etiqueta("B7:C7", "Planificada (ha):");     dato("D7:E7", f"=D{f_tot}", NUM)
    etiqueta("F7", "Lotes atendidos:");         dato("G7:H7")
    etiqueta("B8:C8", "Fecha y Hora Inicio:");  dato("D8:E8")
    etiqueta("F8", "Notificacion: (✔/✘)");     dato("G8:H8")
    etiqueta("B9:C9", "Fecha y Hora Fin:");     dato("D9:E9")
    for col in "FGH":
        dato(f"{col}9")
    for r in range(3, 11):
        ws.row_dimensions[r].height = 20.1

    # --- Cabecera del detalle ----------------------------------------------
    cab = ["N°", "Lote", "Área \n(ha)", "Tipo \nProducto",
           "Producto \n(ml)", "Estado\n(✔/✘)", "Producto \nSobrante (ml)"]
    for col, txt in zip("BCDEFGH", cab):
        rango(ws, f"{col}11", txt, BLANCO_B, VERDE, CENTRO)
    ws.row_dimensions[11].height = 33.95

    # --- Detalle de lotes -------------------------------------------------
    # Lotes planificados: N°, lote y area. Tipo de producto, ml, estado y
    # sobrante los llena el piloto. Filas extra: con formula de ml.
    for i in range(n + FILAS_EXTRA):
        r = ini + i
        ws.row_dimensions[r].height = 21 if i == 0 else 18
        for col in "BCDEFGH":
            rango(ws, f"{col}{r}", None, AZUL, None, CENTRO)
        ws[f"B{r}"].value = i + 1
        for col in "DFH":
            ws[f"{col}{r}"].number_format = NUM
        if i < n:
            ws[f"C{r}"].value = lotes[i]["unidad_05"]
            ws[f"D{r}"].value = round(float(lotes[i]["area"] or 0), 2)
        else:
            ws[f"F{r}"].value = (f'=IF(D{r}="","",D{r}*'
                                 f'IF(E{r}="Trichograma",6,3))')

    dv_prod = DataValidation(type="list", formula1='"Crisopa,Trichograma"',
                             allow_blank=True)
    dv_est = DataValidation(type="list", formula1='"✔,✘"', allow_blank=True)
    ws.add_data_validation(dv_prod)
    ws.add_data_validation(dv_est)
    dv_prod.add(f"E{ini}:E{fin}")
    dv_est.add(f"G{ini}:G{fin}")

    # --- Subtotales (vacios, los llena el piloto) y total de area ---------
    for r, prod in ((f_cri, "Crisopa"), (f_tri, "Trichograma")):
        rango(ws, f"B{r}:C{r}", "Subtotal:", NEGRO_B, VERDE_CLARO, DERECHA)
        rango(ws, f"E{r}", prod, NEGRO_B, VERDE_CLARO, CENTRO)
        for col in "DFGH":
            rango(ws, f"{col}{r}", None, NEGRO_B, VERDE_CLARO, CENTRO,
                  fmt=None if col == "G" else NUM)
    ws.row_dimensions[f_tri].height = 20.1
    rango(ws, f"B{f_tot}:C{f_tot}", "TOTAL GENERAL", BLANCO_B, VERDE, DERECHA)
    rango(ws, f"D{f_tot}", f"=SUM(D{ini}:D{fin})", BLANCO_B, VERDE, CENTRO,
          fmt=NUM)
    for col in "EFGH":
        rango(ws, f"{col}{f_tot}", None, BLANCO_B, VERDE, CENTRO,
              fmt=None if col in "EG" else NUM)

    # --- Observaciones / inconvenientes / firma ----------------------------
    r = f_tot + 2
    for titulo in ("OBSERVACIONES", "INCONVENIENTES"):
        rango(ws, f"B{r}:H{r}", titulo, BLANCO_B, VERDE, IZQ)
        rango(ws, f"B{r+1}:H{r+3}", None, AZUL, None, IZQ_ARRIBA)
        for k in (1, 2, 3):
            ws.row_dimensions[r + k].height = 20.1
        r += 5
    r += 1                               # linea de firma
    ws.merge_cells(f"F{r}:H{r}")
    for col in "FGH":
        ws[f"{col}{r}"].border = Border(bottom=DELGADO)
    rango(ws, f"F{r+1}:H{r+1}", "Firma del Piloto", NEGRO_B, None, CENTRO,
          borde=False)

    # --- Impresion ------------------------------------------------------
    ws.print_area = f"B1:H{r+1}"
    ws.page_setup.orientation = "portrait"
    ws.page_setup.paperSize = ws.PAPERSIZE_LETTER
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 1
    ws.print_options.horizontalCentered = True
    ws.sheet_view.showGridLines = False
    return ws


def construir_libro(filas, codigos, salida):
    por_prop = {}
    for f in filas:
        por_prop.setdefault(int(f["unidad_01"]), []).append(f)

    wb = Workbook()
    wb.remove(wb.active)
    usados, sin_datos = set(), []
    for c in codigos:
        if c in por_prop:
            ws = crear_hoja(wb, por_prop[c], usados)
            print(f"  ✔ {ws.title}: {len(por_prop[c])} lotes")
        else:
            sin_datos.append(c)
    for c in sin_datos:
        print(f"  ✘ {c}: sin lotes en estado {ESTADO}")
    if not wb.sheetnames:
        print("No se generó el Excel: ningún código tiene lotes planificados.")
        return None
    wb.save(salida)
    return salida


def leer_base(codigos):
    try:
        import psycopg2
        import psycopg2.extras
    except ImportError:
        raise ImportError("Falta psycopg2. Ejecuta en una celda:  %pip install psycopg2-binary")
    with psycopg2.connect(**DB) as con:
        with con.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(SQL, (ESTADO, codigos))
            return cur.fetchall()


def generar(codigos, salida=None):
    codigos = list(dict.fromkeys(int(c) for c in codigos))  # sin repetidos
    if not codigos:
        print("La lista CODIGOS está vacía.")
        return None
    salida = salida or f"parte_diario_{dt.date.today():%Y%m%d}_{'_'.join(map(str, codigos))[:60]}.xlsx"
    print(f"Consultando {len(codigos)} propiedad(es)...")
    filas = leer_base(codigos)
    if construir_libro(filas, codigos, salida):
        print(f"Listo: {os.path.abspath(salida)}")
        return salida
    return None


In [3]:
# ---------------------------------------------------------------------------
# EJECUCION: coloca aqui los codigos de propiedad y ejecuta la celda
# ---------------------------------------------------------------------------
CODIGOS = [
310,
1953,
300,
122,
270,
1694,
100,
1831,
2281,
18,
375,
2315,
222,
1585,
55,
288,
1702,
1173,
1255,
2093,
2008,
172,
1771,
2308,
174,
408,
2132,
429,
437,
877,
299,
1513,
1637,
1273,
426,
830,
1131,
2307,
1601,
277,
272,
529,
945

]

In [4]:
SALIDA = 'partes_semana48.xlsx'  # o un nombre, p. ej. "partes_semana45.xlsx"
generar(CODIGOS, SALIDA)

Consultando 43 propiedad(es)...
  ✔ 310 - PUESTO ARENA--JIMENEZ LID: 8 lotes
  ✔ 1953 - TOTAISES--FRANCO GREGORI: 4 lotes
  ✔ 300 - CURICHI GRANDE: 10 lotes
  ✔ 122 - LAS CHACRAS-- WENDE: 29 lotes
  ✔ 270 - POTREROS DEL OBISPO ECOFO: 17 lotes
  ✔ 1694 - EL TRILLO--BALCAZAR RAMO: 12 lotes
  ✔ 100 - RINCONCITO--ESPINOZA: 1 lotes
  ✔ 1831 - SANTA ROSA--PACO ROSA: 1 lotes
  ✔ 2281 - 30 ENERO P51--PAREDES: 2 lotes
  ✔ 18 - LAS MARAS FRERKING: 13 lotes
  ✔ 375 - SANTA ANITA--GUILLERMO SA: 2 lotes
  ✔ 2315 - COLONIA BERLIN I: 3 lotes
  ✔ 222 - POTRERITO: 4 lotes
  ✔ 1585 - EL COLORADO--PROSER SRL: 8 lotes
  ✔ 55 - LA CONQUISTA--AGUILERA: 7 lotes
  ✔ 288 - 5 HERMANOS CUELLAR: 14 lotes
  ✔ 1702 - RANCHO DON JULIO--ROJAS: 2 lotes
  ✔ 1173 - CANAVERAL--FRANCO GREGOR: 4 lotes
  ✔ 1255 - PARCELA 50--GUZMAN JALDI: 2 lotes
  ✔ 2093 - SANTA ROSA--CANDIA: 4 lotes
  ✔ 2008 - OCOROTU: 4 lotes
  ✔ 172 - EL TESORO--ARNEZ Y COLQUE: 3 lotes
  ✔ 1771 - FJA ILLIMANI 09--AGRODAR: 1 lotes
  ✔ 2308 - EL CAÑAVERAL

'partes_semana48.xlsx'